In [2]:
import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import org.apache.spark.sql.{Dataset, DataFrame, Row, SparkSession}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._

val spark = SparkSession.builder()
  .appName("Clase20")
  .master("local[*]")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()

import spark.implicits._

spark.sparkContext.setLogLevel("ERROR")

println(s"✅ Spark${spark.version} listo")
println(s"✅ Scala${scala.util.Properties.versionNumberString}")
println(s"✅ Java${System.getProperty("java.version")}")

✅ Spark4.1.1 listo
✅ Scala2.13.18
✅ Java17.0.12


import $ivy.$
import $ivy.$
import org.apache.spark.sql.{Dataset, DataFrame, Row, SparkSession}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@5e20fcba
import spark.implicits._

In [3]:
case class Cliente(id: Int, nombre: String, ciudad: String, edad: Int)

defined class Cliente

In [4]:
val dsClientes: Dataset[Cliente] = Seq(
  Cliente(1, "Ana", "Madrid", 28),
  Cliente(2, "Luis", "Barcelona", 34),
  Cliente(3, "Marta", "Sevilla", 22)
).toDS()

dsClientes.show(false)

+---+------+---------+----+
|id |nombre|ciudad   |edad|
+---+------+---------+----+
|1  |Ana   |Madrid   |28  |
|2  |Luis  |Barcelona|34  |
|3  |Marta |Sevilla  |22  |
+---+------+---------+----+



dsClientes: Dataset[Cliente] = [id: int, nombre: string ... 2 more fields]

Ejercicio 1 — Crear un Dataset tipado de empleados
Enunciado
Crea un Dataset tipado llamado dsEmpleados usando una case class llamada Empleado.

Datos de entrada
Celda 1 — Definir la case class

In [5]:
case class Empleado(
  id: Int,
  nombre: String,
  departamento: String,
  salario: Double,
  activo: Boolean
)

defined class Empleado

In [6]:
//Celda 2 — Crear el Dataset
val dsEmpleados = Seq(
  Empleado(1, "Ana García", "Ingeniería", 42000.0, true),
  Empleado(2, "Luis Martínez", "Marketing", 38000.0, true),
  Empleado(3, "Marta López", "Ingeniería", 35000.0, true),
  Empleado(4, "Pedro Ruiz", "Dirección", 75000.0, true),
  Empleado(5, "Carmen Díaz", "Marketing", 36500.0, true),
  Empleado(6, "Jorge Santos", "Ingeniería", 48000.0, false)
).toDS()

dsEmpleados: Dataset[Empleado] = [id: int, nombre: string ... 3 more fields]

Tareas
Mostrar el contenido con show(false).
Imprimir el schema.
Filtrar solo empleados activos.
Filtrar empleados con salario mayor o igual a 40.000.

In [7]:
println("=== Empleados ===")
dsEmpleados.show(false)

println("=== Schema ===")
dsEmpleados.printSchema()

val empleadosActivos = dsEmpleados.filter(e => e.activo)

println("=== Empleados activos ===")
empleadosActivos.show(false)

val empleadosSalarioAlto = dsEmpleados.filter(e => e.salario >= 40000.0)

println("=== Empleados con salario >= 40000 ===")
empleadosSalarioAlto.show(false)

=== Empleados ===
+---+-------------+------------+-------+------+
|id |nombre       |departamento|salario|activo|
+---+-------------+------------+-------+------+
|1  |Ana García   |Ingeniería  |42000.0|true  |
|2  |Luis Martínez|Marketing   |38000.0|true  |
|3  |Marta López  |Ingeniería  |35000.0|true  |
|4  |Pedro Ruiz   |Dirección   |75000.0|true  |
|5  |Carmen Díaz  |Marketing   |36500.0|true  |
|6  |Jorge Santos |Ingeniería  |48000.0|false |
+---+-------------+------------+-------+------+

=== Schema ===
root
 |-- id: integer (nullable = false)
 |-- nombre: string (nullable = true)
 |-- departamento: string (nullable = true)
 |-- salario: double (nullable = false)
 |-- activo: boolean (nullable = false)

=== Empleados activos ===
+---+-------------+------------+-------+------+
|id |nombre       |departamento|salario|activo|
+---+-------------+------------+-------+------+
|1  |Ana García   |Ingeniería  |42000.0|true  |
|2  |Luis Martínez|Marketing   |38000.0|true  |
|3  |Marta López

empleadosActivos: Dataset[Empleado] = [id: int, nombre: string ... 3 more fields]
empleadosSalarioAlto: Dataset[Empleado] = [id: int, nombre: string ... 3 more fields]

Ejercicio 2 — Crear una nueva estructura con map
Enunciado
A partir de dsEmpleados, crea un nuevo Dataset llamado dsEmpleadosClasificados, donde cada empleado tenga una categoría salarial.

Reglas:

Salario	Categoría
>= 60000	ALTO
>= 40000 y < 60000	MEDIO
< 40000	     BAJO

In [8]:
case class EmpleadoClasificado(
  id: Int,
  nombre: String,
  departamento: String,
  salario: Double,
  activo: Boolean,
  categoriaSalarial: String
)

defined class EmpleadoClasificado

In [9]:
val dsEmpleadosClasificados = dsEmpleados.map { e =>
  val categoria =
    if (e.salario >= 60000.0) "ALTO"
    else if (e.salario >= 40000.0) "MEDIO"
    else "BAJO"

  EmpleadoClasificado(
    e.id,
    e.nombre,
    e.departamento,
    e.salario,
    e.activo,
    categoria
  )
}

dsEmpleadosClasificados.show(false)

+---+-------------+------------+-------+------+-----------------+
|id |nombre       |departamento|salario|activo|categoriaSalarial|
+---+-------------+------------+-------+------+-----------------+
|1  |Ana García   |Ingeniería  |42000.0|true  |MEDIO            |
|2  |Luis Martínez|Marketing   |38000.0|true  |BAJO             |
|3  |Marta López  |Ingeniería  |35000.0|true  |BAJO             |
|4  |Pedro Ruiz   |Dirección   |75000.0|true  |ALTO             |
|5  |Carmen Díaz  |Marketing   |36500.0|true  |BAJO             |
|6  |Jorge Santos |Ingeniería  |48000.0|false |MEDIO            |
+---+-------------+------------+-------+------+-----------------+



dsEmpleadosClasificados: Dataset[EmpleadoClasificado] = [id: int, nombre: string ... 4 more fields]

Ejercicio 3 — Convertir Dataset a DataFrame
Enunciado
Convierte dsEmpleadosClasificados a DataFrame y realiza una agregación por departamento.

In [10]:
val dfEmpleadosClasificados = dsEmpleadosClasificados.toDF()

val resumenPorDepartamento = dfEmpleadosClasificados
  .groupBy("departamento")
  .agg(
    count("id").alias("num_empleados"),
    avg("salario").alias("salario_medio"),
    max("salario").alias("salario_maximo")
  )

resumenPorDepartamento.show(false)

+------------+-------------+------------------+--------------+
|departamento|num_empleados|salario_medio     |salario_maximo|
+------------+-------------+------------------+--------------+
|Ingeniería  |3            |41666.666666666664|48000.0       |
|Marketing   |2            |37250.0           |38000.0       |
|Dirección   |1            |75000.0           |75000.0       |
+------------+-------------+------------------+--------------+



dfEmpleadosClasificados: DataFrame = [id: int, nombre: string ... 4 more fields]
resumenPorDepartamento: DataFrame = [departamento: string, num_empleados: bigint ... 2 more fields]

Ejercicio 4 — Convertir DataFrame a Dataset con .as[T]
Enunciado
Crea un DataFrame de productos y conviértelo a Dataset usando una case class.

In [11]:
val dfProductos = Seq(
  (101, "Laptop Pro", "Informática", 1299.99, 45),
  (102, "Teclado Inalámbrico", "Periféricos", 59.90, 120),
  (103, "Monitor 27", "Monitores", 349.00, 30),
  (104, "Ratón Óptico", "Periféricos", 24.95, 200)
).toDF("id", "nombre", "categoria", "precio", "stock")

dfProductos: DataFrame = [id: int, nombre: string ... 3 more fields]

In [12]:
case class Producto(
  id: Int,
  nombre: String,
  categoria: String,
  precio: Double,
  stock: Int
)

defined class Producto

In [13]:
val dsProductos = dfProductos.as[Producto]

dsProductos.show(false)

dsProductos.printSchema()

+---+-------------------+-----------+-------+-----+
|id |nombre             |categoria  |precio |stock|
+---+-------------------+-----------+-------+-----+
|101|Laptop Pro         |Informática|1299.99|45   |
|102|Teclado Inalámbrico|Periféricos|59.9   |120  |
|103|Monitor 27         |Monitores  |349.0  |30   |
|104|Ratón Óptico       |Periféricos|24.95  |200  |
+---+-------------------+-----------+-------+-----+

root
 |-- id: integer (nullable = false)
 |-- nombre: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- precio: double (nullable = false)
 |-- stock: integer (nullable = false)



dsProductos: Dataset[Producto] = [id: int, nombre: string ... 3 more fields]

In [14]:
//FILTRO TIPADO 
val productosStockAlto = dsProductos.filter(p => p.stock >= 100)

productosStockAlto.show(false)

+---+-------------------+-----------+------+-----+
|id |nombre             |categoria  |precio|stock|
+---+-------------------+-----------+------+-----+
|102|Teclado Inalámbrico|Periféricos|59.9  |120  |
|104|Ratón Óptico       |Periféricos|24.95 |200  |
+---+-------------------+-----------+------+-----+



productosStockAlto: Dataset[Producto] = [id: int, nombre: string ... 3 more fields]

Ejercicio 5 — Comparar error de DataFrame vs Datase

In [15]:
//Parte A — Error con DataFrame
val dfError = dsEmpleados.toDF()

// La columna "departamentoo" no existe
val resultadoErrorDF = dfError.select("nombre", "departamentoo")

resultadoErrorDF.show(false)

org.apache.spark.sql.catalyst.ExtendedAnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `departamentoo` cannot be resolved. Did you mean one of the following? [`departamento`, `activo`, `salario`, `nombre`, `id`]. SQLSTATE: 42703;
'Project [nombre#36, 'departamentoo]
+- LocalRelation [id#35, nombre#36, departamento#37, salario#38, activo#39]


In [16]:
//Parte B — Error con Dataset
// Este código NO compila porque departamentoo no existe en la case class Empleado
val resultadoErrorDS = dsEmpleados.map(e => e.departamentoo)

cmd16.sc:3: value departamentoo is not a member of ammonite.$sess.cmd6.wrapper.cmd5.Empleado
did you mean departamento?
val resultadoErrorDS = dsEmpleados.map(e => e.departamentoo)
                                              ^
Compilation Failed

Pregunta para reflexionar
¿Qué error preferirías encontrar en un proyecto real: uno al compilar o uno después de lanzar el job Spark?

R= Siempre es preferible encontrar el error al compilar.En un proyecto real, un error de compilación es "barato": se detecta en segundos en el entorno de desarrollo y no consume recursos. En cambio, un error en tiempo de ejecución (runtime) tras lanzar el job de Spark es "caro" por varias razones:Pérdida de tiempo: El job puede fallar tras horas de procesamiento, obligándote a empezar de cero.Coste económico: Estás pagando por tiempo de cómputo en el clúster (AWS, Azure, GCP) para un proceso que acabará en fallo.Inconsistencia de datos: Un fallo a mitad de una escritura puede dejar datos parciales o corruptos en tu Data Lake si no gestionas bien la atomicidad.Por eso, el uso de Datasets frente a DataFrames es tan valorado en Scala; "mueves" el riesgo del servidor a tu propia máquina antes de desplegar.

🧪 Pipeline completo con Dataset tipado
Enunciado
Vas a construir un pipeline completo de ventas usando Datasets tipados.

El objetivo es:

Crear un Dataset de ventas.
Filtrar ventas válidas.
Calcular el total de cada venta.
Clasificar cada venta según su importe.
Convertir el resultado a DataFrame.
Agregar ventas por categoría

In [16]:
case class VentaRaw(
  idVenta: Int,
  producto: String,
  categoria: String,
  cantidad: Int,
  precioUnitario: Double,
  ciudad: String
)

case class VentaProcesada(
  idVenta: Int,
  producto: String,
  categoria: String,
  cantidad: Int,
  precioUnitario: Double,
  ciudad: String,
  total: Double,
  tipoVenta: String
)

defined class VentaRaw
defined class VentaProcesada

In [17]:
val dsVentasRaw = Seq(
  VentaRaw(1, "Portátil", "Informática", 2, 850.50, "Madrid"),
  VentaRaw(2, "Ratón", "Informática", 5, 18.90, "Valencia"),
  VentaRaw(3, "Teclado", "Informática", 3, 45.00, "Sevilla"),
  VentaRaw(4, "Monitor", "Informática", 1, 199.99, "Madrid"),
  VentaRaw(5, "Silla", "Oficina", 4, 120.00, "Barcelona"),
  VentaRaw(6, "Mesa", "Oficina", 2, 250.00, "Zaragoza"),
  VentaRaw(7, "Webcam", "Informática", 6, 39.90, "Madrid"),
  VentaRaw(8, "Auriculares", "Informática", 3, 59.99, "Valencia"),
  VentaRaw(9, "Producto defectuoso", "Prueba", 0, 100.00, "Madrid")
).toDS()

println("=== Ventas raw ===")
dsVentasRaw.show(false)

=== Ventas raw ===
+-------+-------------------+-----------+--------+--------------+---------+
|idVenta|producto           |categoria  |cantidad|precioUnitario|ciudad   |
+-------+-------------------+-----------+--------+--------------+---------+
|1      |Portátil           |Informática|2       |850.5         |Madrid   |
|2      |Ratón              |Informática|5       |18.9          |Valencia |
|3      |Teclado            |Informática|3       |45.0          |Sevilla  |
|4      |Monitor            |Informática|1       |199.99        |Madrid   |
|5      |Silla              |Oficina    |4       |120.0         |Barcelona|
|6      |Mesa               |Oficina    |2       |250.0         |Zaragoza |
|7      |Webcam             |Informática|6       |39.9          |Madrid   |
|8      |Auriculares        |Informática|3       |59.99         |Valencia |
|9      |Producto defectuoso|Prueba     |0       |100.0         |Madrid   |
+-------+-------------------+-----------+--------+--------------+----

dsVentasRaw: Dataset[VentaRaw] = [idVenta: int, producto: string ... 4 more fields]

In [18]:
//Paso 3 — Filtrar ventas válidas
val dsVentasValidas = dsVentasRaw.filter { venta =>
  venta.cantidad > 0 && venta.precioUnitario > 0
}

println("=== Ventas válidas ===")
dsVentasValidas.show(false)

=== Ventas válidas ===
+-------+-----------+-----------+--------+--------------+---------+
|idVenta|producto   |categoria  |cantidad|precioUnitario|ciudad   |
+-------+-----------+-----------+--------+--------------+---------+
|1      |Portátil   |Informática|2       |850.5         |Madrid   |
|2      |Ratón      |Informática|5       |18.9          |Valencia |
|3      |Teclado    |Informática|3       |45.0          |Sevilla  |
|4      |Monitor    |Informática|1       |199.99        |Madrid   |
|5      |Silla      |Oficina    |4       |120.0         |Barcelona|
|6      |Mesa       |Oficina    |2       |250.0         |Zaragoza |
|7      |Webcam     |Informática|6       |39.9          |Madrid   |
|8      |Auriculares|Informática|3       |59.99         |Valencia |
+-------+-----------+-----------+--------+--------------+---------+



dsVentasValidas: Dataset[VentaRaw] = [idVenta: int, producto: string ... 4 more fields]

In [19]:
//Paso 4 — Calcular total y clasificar ventas
val dsVentasProcesadas = dsVentasValidas.map { venta =>
  val total = venta.cantidad * venta.precioUnitario

  val tipoVenta =
    if (total >= 1000.0) "VENTA_ALTA"
    else if (total >= 300.0) "VENTA_MEDIA"
    else "VENTA_BAJA"

  VentaProcesada(
    venta.idVenta,
    venta.producto,
    venta.categoria,
    venta.cantidad,
    venta.precioUnitario,
    venta.ciudad,
    total,
    tipoVenta
  )
}

println("=== Ventas procesadas ===")
dsVentasProcesadas.show(false)

=== Ventas procesadas ===
+-------+-----------+-----------+--------+--------------+---------+------------------+-----------+
|idVenta|producto   |categoria  |cantidad|precioUnitario|ciudad   |total             |tipoVenta  |
+-------+-----------+-----------+--------+--------------+---------+------------------+-----------+
|1      |Portátil   |Informática|2       |850.5         |Madrid   |1701.0            |VENTA_ALTA |
|2      |Ratón      |Informática|5       |18.9          |Valencia |94.5              |VENTA_BAJA |
|3      |Teclado    |Informática|3       |45.0          |Sevilla  |135.0             |VENTA_BAJA |
|4      |Monitor    |Informática|1       |199.99        |Madrid   |199.99            |VENTA_BAJA |
|5      |Silla      |Oficina    |4       |120.0         |Barcelona|480.0             |VENTA_MEDIA|
|6      |Mesa       |Oficina    |2       |250.0         |Zaragoza |500.0             |VENTA_MEDIA|
|7      |Webcam     |Informática|6       |39.9          |Madrid   |239.399999999999

dsVentasProcesadas: Dataset[VentaProcesada] = [idVenta: int, producto: string ... 6 more fields]

In [20]:
//Paso 5 — Guardar el resultado del pipeline en una variable
val pipelineVentasTipado = dsVentasProcesadas

pipelineVentasTipado.show(false)

+-------+-----------+-----------+--------+--------------+---------+------------------+-----------+
|idVenta|producto   |categoria  |cantidad|precioUnitario|ciudad   |total             |tipoVenta  |
+-------+-----------+-----------+--------+--------------+---------+------------------+-----------+
|1      |Portátil   |Informática|2       |850.5         |Madrid   |1701.0            |VENTA_ALTA |
|2      |Ratón      |Informática|5       |18.9          |Valencia |94.5              |VENTA_BAJA |
|3      |Teclado    |Informática|3       |45.0          |Sevilla  |135.0             |VENTA_BAJA |
|4      |Monitor    |Informática|1       |199.99        |Madrid   |199.99            |VENTA_BAJA |
|5      |Silla      |Oficina    |4       |120.0         |Barcelona|480.0             |VENTA_MEDIA|
|6      |Mesa       |Oficina    |2       |250.0         |Zaragoza |500.0             |VENTA_MEDIA|
|7      |Webcam     |Informática|6       |39.9          |Madrid   |239.39999999999998|VENTA_BAJA |
|8      |A

pipelineVentasTipado: Dataset[VentaProcesada] = [idVenta: int, producto: string ... 6 more fields]

In [21]:
//Paso 6 — Convertir a DataFrame para agregaciones
val dfVentasProcesadas = pipelineVentasTipado.toDF()

val resumenCategoria = dfVentasProcesadas
  .groupBy("categoria")
  .agg(
    count("idVenta").alias("num_ventas"),
    sum("total").alias("importe_total"),
    avg("total").alias("importe_medio")
  )
  .orderBy(desc("importe_total"))

resumenCategoria.show(false)

+-----------+----------+------------------+------------------+
|categoria  |num_ventas|importe_total     |importe_medio     |
+-----------+----------+------------------+------------------+
|Informática|6         |2549.8599999999997|424.97666666666663|
|Oficina    |2         |980.0             |490.0             |
+-----------+----------+------------------+------------------+



dfVentasProcesadas: DataFrame = [idVenta: int, producto: string ... 6 more fields]
resumenCategoria: Dataset[Row] = [categoria: string, num_ventas: bigint ... 2 more fields]